# The CHORD GNSS beam cube

A measured beam pattern for each dish, built from GNSS satellites crossing the sky. Each
satellite is a point source of known position, so every tracked satellite-second is one
sample of the response in whatever direction it happened to be.

This notebook loads one day, makes a sky map, and slices it by element and frequency. It
also flags the four things that are easy to get wrong -- please read those before drawing
conclusions from a map.

**Needs** `numpy`, `healpy`, `matplotlib`. **Runtime** well under a minute.


## Where the data is

One `.npz` per UTC day per resolution:

```
/home/kvand/gnss/fixtures/beamcube/cube_<YYYYMMDD>_nside64.npz    # 0.3-1.1 GB
/home/kvand/gnss/fixtures/beamcube/cube_<YYYYMMDD>_nside128.npz   # 0.2-0.9 GB
```

`nside 64` is 0.92 deg pixels, `nside 128` is 0.46 deg. Start with 64 -- see caveat 3.
There is also a browser view of the same data at <http://cf06:8877>.

In [ ]:
import json

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

DAY = "20260907"
PATH = f"/home/kvand/gnss/fixtures/beamcube/cube_{DAY}_nside64.npz"

# The telescope's fixed pointing, in the local horizon frame. Nothing in the
# pipeline is told this: the beam is centred here because the satellites put it
# there, which is a check on the whole chain.
BORE_AZ, BORE_EL = 180.0, 81.41

## Loading

The file stores each chain sparsely -- only the pixels that were actually visited -- so
this expands them to full healpix maps. Everything is `RING` ordering.

In [ ]:
def load_beam_cube(path, chain=None):
    """-> {'meta': ..., 'chains': {name: {'n','s1','s2','freq_ids'}}}

    n, s1, s2 are dense [n_subband, n_element, npix] with npix = 12*nside**2.
    """
    z = np.load(path, allow_pickle=False)
    meta = json.loads(str(z["meta"]))
    npix = 12 * meta["nside"] ** 2
    out = {}
    for i, c in enumerate(meta["chains"]):
        if chain and c["chain"] != chain:
            continue
        pix = z[f"pix_{i}"]
        d = {"freq_ids": [f[0] for f in c["freq_ids"]]}
        for q, dt in (("n", np.int64), ("s1", np.float64), ("s2", np.float64)):
            a = np.zeros((c["n_sub"], c["n_elem"], npix), dt)
            a[:, :, pix] = z[f"{q}_{i}"]
            d[q] = a
        out[c["chain"]] = d
    return {"meta": meta, "chains": out}


cube = load_beam_cube(PATH, chain="gps_l5")
meta = cube["meta"]
l5 = cube["chains"]["gps_l5"]
nside = meta["nside"]

print(f"day {meta['day']}  nside {nside}  units {meta['units']}  pointing {meta['pointing']}")
print(f"gps_l5 n: {l5['n'].shape}  = (subband, element, healpix)")
print(f"channels {l5['freq_ids'][0]}..{l5['freq_ids'][-1]}  "
      f"({l5['freq_ids'][0] * 0.1953125:.1f}-{l5['freq_ids'][-1] * 0.1953125:.1f} MHz)")

Other chains in the same file are the other GNSS signals: `gal_e5a`, `bds_b2a` (both also
1176 MHz, so directly comparable with `gps_l5`), plus `gal_e5b`, `bds_b2b`, `bds_b3i`,
`gal_e6`, `gps_l2c` at other frequencies. Pass `chain=None` to load them all -- that is
several GB in memory, so load what you need.

## Caveat 1 -- these are accumulators, not values

Each cell holds `n` (how many samples landed there), `s1` (their sum) and `s2` (sum of
squares). The measurement is `s1/n`; `s2` gives you the scatter. Storing it this way is what
makes every collapse meaningful -- sum the accumulators over any axis, *then* divide.

**Take dB last.** Decibels cannot be summed or averaged, so `10*log10(sum(s1)/sum(n))` is
right and `mean(10*log10(s1/n))` is not.

The units are *pedestal*: every sample was divided by that element's own noise floor in that
channel before being accumulated. So a cell is "power over this element's own noise here" --
dimensionless, and comparable across elements, channels and chains.

In [ ]:
def sky_map(d, subbands=slice(None), elements=slice(None), min_n=8):
    """Collapse (subband, element) -> one healpix map of mean power. NaN where unsampled."""
    n = d["n"][subbands, elements].sum(axis=(0, 1))
    s1 = d["s1"][subbands, elements].sum(axis=(0, 1))
    with np.errstate(all="ignore"):
        return np.where(n >= min_n, s1 / np.maximum(n, 1), np.nan)


power = sky_map(l5)
hit = np.isfinite(power) & (power > 0)
db = np.full_like(power, np.nan)
db[hit] = 10 * np.log10(power[hit])

print(f"{hit.sum()} of {power.size} pixels sampled ({100 * hit.mean():.1f}% of the sphere)")
print(f"peak {np.nanmax(db):.1f} dB, median {np.nanmedian(db):.1f} dB")

## Caveat 2 -- the pixelisation is the LOCAL HORIZON frame, not the sky

`theta = 90 - elevation`, `phi = azimuth` (north = 0, increasing clockwise). This is
deliberate: the beam is fixed to the dish, so a celestial pixelisation would smear a
stationary pattern across the map as the earth turns.

So a pixel is a direction *relative to the ground*, and the fixed pointing sits at a fixed
place in every map.

In [ ]:
hp.gnomview(db, rot=(BORE_AZ, BORE_EL), reso=3.0, xsize=420,
            min=np.nanpercentile(db, 60), max=np.nanmax(db),
            title=f"gps_l5, all channels, all elements ({meta['day']})",
            unit="dB (pedestal units)", notext=True)
hp.graticule(dpar=5, dmer=5, local=True)
plt.show()

# Where is the brightest surviving pixel? NOT at the centre -- every epoch with a
# satellite inside 5 deg of the pointing is vetoed (caveat 3), so the main lobe is
# cut out and the peak sits on the rim of that hole.
theta, phi = hp.pix2ang(nside, int(np.nanargmax(db)))
az_pk, el_pk = np.degrees(phi), 90 - np.degrees(theta)
sep = np.degrees(np.arccos(np.clip(
    np.sin(np.radians(el_pk)) * np.sin(np.radians(BORE_EL))
    + np.cos(np.radians(el_pk)) * np.cos(np.radians(BORE_EL))
      * np.cos(np.radians(az_pk - BORE_AZ)), -1, 1)))
print(f"pointing:         az {BORE_AZ:.1f}  el {BORE_EL:.1f}")
print(f"brightest pixel:  az {az_pk:.1f}  el {el_pk:.1f}  -> {sep:.1f} deg out")
print("expected just outside the 5 deg veto radius, and it is")

## Slicing by element and by frequency

The element axis is one entry per feed; five are dark and read ~18 dB low. The subband axis
is absolute channel numbers -- `freq_ids[k] * 0.1953125` MHz -- so the same index means the
same sky frequency in every chain that covers it.

In [ ]:
# One element, and one channel, on the same colour scale as each other.
fid = 5988
k = l5["freq_ids"].index(fid)

for label, m in (("element 0, all channels", sky_map(l5, elements=slice(0, 1))),
                 (f"all elements, freq_id {fid}", sky_map(l5, subbands=slice(k, k + 1)))):
    d = np.full_like(m, np.nan)
    ok = np.isfinite(m) & (m > 0)
    d[ok] = 10 * np.log10(m[ok])
    hp.gnomview(d, rot=(BORE_AZ, BORE_EL), reso=3.0, xsize=360, title=label,
                min=np.nanpercentile(d, 60), max=np.nanmax(d), unit="dB", notext=True)
plt.show()

### A radial cut

Averaging in rings around the pointing is the quickest way to see the structure.

The ripple is diffraction, and its spacing is worth a second look: it comes out near
5 deg, which is `lambda/L` for **L ~ 3 m** -- half the 6 m dish -- and it scales with
wavelength across the bands, so it is geometry rather than an artefact of the sampling.
Whether that means an underilluminated aperture or a two-path interference at the dish
radius is not settled; the measurement that would separate them needs the main lobe,
which is exactly what the veto removes.

In [ ]:
def angsep_deg(az, el, az0, el0):
    a, e, a0, e0 = map(np.radians, (az, el, az0, el0))
    c = np.sin(e) * np.sin(e0) + np.cos(e) * np.cos(e0) * np.cos(a - a0)
    return np.degrees(np.arccos(np.clip(c, -1, 1)))


th_pix, ph_pix = hp.pix2ang(nside, np.arange(power.size))
theta_deg = angsep_deg(np.degrees(ph_pix), 90 - np.degrees(th_pix), BORE_AZ, BORE_EL)

edges = np.arange(5, 40.1, 1.0)
centres, profile = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    sel = (theta_deg >= lo) & (theta_deg < hi) & np.isfinite(db)
    if sel.sum() >= 3:
        centres.append(0.5 * (lo + hi))
        profile.append(np.median(db[sel]))

plt.figure(figsize=(7, 4))
plt.plot(centres, profile, "-o", ms=3)
plt.xlabel("angle from pointing [deg]")
plt.ylabel("dB (pedestal units)")
plt.title(f"gps_l5 radial cut, {meta['day']}")
plt.grid(alpha=0.3)
plt.show()

## Caveat 3 -- a map is TRACK-limited, not resolution-limited

The sky is sampled only where satellites went. At `nside 64` one day covers a few per cent
of the sphere; at `nside 128` the same tracks are spread over four times as many pixels and
the map looks moth-eaten. Finer pixels do not buy resolution unless you also add days.

Two consequences worth knowing:

* **Coverage saturates at about 79%** of the visible sky after ~10 days. The missing fifth is
  the north: no GNSS satellite goes there from this latitude. More nights buy depth, never
  coverage.
* **The main lobe is missing.** A satellite within 5 deg of the pointing saturates the 4+4-bit
  requantiser for every chain at once, so those epochs are vetoed -- which removes the very
  brightest part of the beam. The hole at the centre of the map is that veto, not a null.

## Caveat 4 -- there is no time axis here

One file is one UTC day, already integrated. If you need time resolution, the source is the
L0 archive: `/mnt/cs00/data/kvand/gnss_cube/l0/<pointing>/<sender>/<day>.h5`, one HDF5 per
sender per day at roughly 1-second windows -- 720 GB, with 12x and 60x pre-summed ladders
alongside at 73 GB and 15 GB. That is a different (and heavier) exercise than this notebook;
ask and we can point you at the row schema.

---

Questions, or a slice you want that is awkward here: ask Keith.